# PopOut Game - Monte Carlo Tree Search and ID3 Decision Tree

Project done by:
- Ana Francisca Araújo
- Sofia Ribeiro
- Pedro Jorge

## 1. Project overview

This project, carried out within the scope of the Artificial Intelligence course, implements a complete AI pipeline for the PopOut board game, it consists of three major components:
- A PopOut game bitboard-based implementation
- A Monte Carlo Tree Search that searches for optimal moves during play
- An ID3 Decision Tree that infers the best action based on a game state, from MCTS-generated game data

## 2. Popout Implementation

For the sake of computation efficiency, we decided to implement the PopOut game in a bitboard, instead of the conventional 2-dimentional matrix. In that way, every possible action whithin the rules of the game will be converted in bitwise operations.

#### 2.1. Bitboard representation
The board state is stored as two 64-bit integers, one for the current player and one for the opponent. Each column occupies 7 bits (0 - 5 fot the 6 rows, with the 6th bit being a filler bit to prevent horizontal overflow). The layout is:

- Bit positions: [col6: bits 42-48] [col5: bits 35-41] ... [col0: bits 0-5]

- Within a column: bit 0 = row 0, bit 5 = row 5, bit 6 = filler

Important bitboard operations:
- `player | opponent | filler_mask`: Mask of all occupied bits
- `occupied & col_mask`: Empty cells in a certain column
- `empty_in_col & -empty_in_col`: Lowest set bit
- `bitboard & (bittboard << shift)`: Pairs of matching pieces in a direction

#### 2.2. Zobrist Class

Zobrist hashing is a technique for efficiently computing unqiue hash of a board position. Each (cell, player) pair is assigner a random 64-bit integer at initialisation. The has of any position is the XOR of all random values corresponding to the pieces currently on the board, plus an aditional XOR for whose turn it is. 

In [ ]:
from random import getrandbits


class Zobrist:
    def __init__(self):
        # 42 cells on board x 2 player (player id 0, player id 1)
        self.zArray = [[getrandbits(64) for _ in range(2)] for _ in range(42)]
        self.zTurn = [getrandbits(64), getrandbits(64)]  # [player id i]

    def _compute_hash(self, player, opponent, cur_player):
        """
        Args:
            player: current player bitboard
            opponent: opponent bitboard
            cur_player: ID of player
        """
        h = 0
        other_player = 1 - cur_player

        for col in range(7):
            for row in range(6):
                bit_index = col * 7 + row  # skip filler bit at col*7 + 6
                cell = col * 6 + row       # flat 42-cell index for Zobrist table
                mask = 1 << bit_index

                if player & mask:
                    h ^= self.zArray[cell][cur_player]
                elif opponent & mask:
                    h ^= self.zArray[cell][other_player]

        h ^= self.zTurn[cur_player]
        return h

    def _mirror(self, bitboard: int):
        """Reflect columns: col 0 <-> col 6, col 1 <-> col 5, col 2 <-> col 4, col 3 stays"""
        result = 0
        for col in range(7):
            mirrored_col = 6 - col
            # Extract col's 6 bits and place them at mirrored_col's position
            col_bits = (bitboard >> (col * 7)) & 0x3F  # 0x3F = 0b111111 (6 bits)
            result |= col_bits << (mirrored_col * 7)
        return result

    def get_hash(self, player, opponent, cur_player):
        """
        Returns:
            (isMirrored, Canonical hash)
        """
        h_normal = self._compute_hash(player, opponent, cur_player)
        h_mirror = self._compute_hash(self._mirror(player), self._mirror(opponent), cur_player)
        if h_mirror < h_normal:
            return True, h_mirror
        return False, h_normal


In this class we mantained two data structures:

- zArray: a 42 x 2 table of 64-bit integers, indexed by (flat cell index, player id)

- zTurn: a 2 element array of random integers, one per player, XORed in to encode whose turn it is

Imporant class methods:

- `_compute_hash(player, opponent, cur_player)` → int

Iterates over all the 42 cells, and for each one, it converts the bitboard position (col * 7 + row) to a flat index (col * 6 + row), checks if it is occupied and XORs in the corresponding entry from zArray (player or opponent). Finnaly, it XORs in zTurn[current_player]. The result is a 64-bit integer that uniquely identifies the position.

- `_mirror(bitboard)` → int

Reflects the board left to right by swapping collumns: col0 ↔ col6, col1 ↔ col5, col2 ↔ col4. For each collumn it extracts the 6 data bits using (bitboard >> (col * 7)) & 0x3F and places them at position (mirrorred_col * 7) in the result.

- `_get_hash(player, opponent, cur_player)` → int

Computes both the normal and the mirrored hash, and return the smallest of the two as the canonical hash. This ensures that a board and its horizontal mirror are treated as the same style, halving the effective search space.

#### 2.3. PopOut Class

This class is the central game state representation, with all game logic defined. It initializes the two bitboards to 0, builds the filler_mask by setting bit 6 of each of the 7 collumns, records the initial (empty) state in  state_counts and assigns the first player.

In [ ]:
from collections import defaultdict

ROWS = 6
COLS = 7

Action = tuple[str, int]
class PopOut:
    def __init__(self, first_player: int = 0):
        """
        Each bitboard is of the format:
            [col6]0[col5]0[col4]0[col3]0[col2]0[col1]0[col0]

        Note: between each [col_i] and [col_j] there is a filler bit;
              the right-most bit in [col_i] represents row0, and the
              left-most bit represents row5
        """
        # Create filler mask (bit 6 of each column)
        self.filler_mask = 0
        for col in range(7):
            self.filler_mask |= (1 << (col * 7 + 6))  # bit 6 of each column

        # Board size is 6x7 = 42 bits + 6 filler bits
        self.player = 0    # Current player's board
        self.opponent = 0  # Current opponent's board

        self.cur_player = 0  # ID: 0 or 1
        self.zobrist = Zobrist()
        self.state_counts = defaultdict(int)

        self._record_state()  # Record initial state

    def __str__(self):
        display = [['-' for _ in range(COLS)] for _ in range(ROWS)]
        for bit in range(48):
            col = bit // 7
            row = bit % 7
            if row >= ROWS or col >= COLS:
                continue
            if (self.player >> bit) & 1:
                display_row = ROWS - 1 - row
                display[display_row][col] = 'O'
            elif (self.opponent >> bit) & 1:
                display_row = ROWS - 1 - row
                display[display_row][col] = 'X'
        return '\n'.join(' '.join(row) for row in display)

    def copy(self):
        """Create a lightweight copy with only essential attributes."""
        new = PopOut.__new__(PopOut)
        new.filler_mask = self.filler_mask
        new.player = self.player
        new.opponent = self.opponent
        new.cur_player = self.cur_player
        new.zobrist = self.zobrist
        new.state_counts = self.state_counts.copy()
        # new.state_counts = defaultdict(int)
        return new

    def get_hash(self) -> int:
        """Returns Canonical Hash"""
        _, h = self.zobrist.get_hash(self.player, self.opponent, self.cur_player)
        return h

    def get_raw_hash(self) -> int:
        return self.zobrist._compute_hash(self.player, self.opponent, self.cur_player)

    def get_hash_with_mirror(self) -> tuple[bool, int]:
        """Returns (isMirrored, Canonical Hash)"""
        return self.zobrist.get_hash(self.player, self.opponent, self.cur_player)

    def _record_state(self) -> None:
        self.state_counts[self.get_raw_hash()] += 1

    def is_threefold_repetition(self) -> bool:
        return self.state_counts[self.get_raw_hash()] >= 3

    def switch_turn(self) -> None:
        self.player, self.opponent = self.opponent, self.player
        self.cur_player = 1 - self.cur_player

    def get_occupied(self) -> int:
        return (self.player | self.opponent) | self.filler_mask

    def get_empty(self) -> int:
        return ~self.get_occupied() & 0x3FFFFFFFFFF  # 42 bit mask

    def win(self, opponent=False) -> bool:
        """
        Check if player wins.

        Args:
            opponent: Enable win check of opponent instead
        """
        bitboard = self.player
        if opponent:
            bitboard = self.opponent

        # All directions: horizontal (1), vertical (7),
        # diagonal down-right (6), diagonal up-right (8)
        for shift in [1, 7, 6, 8]:
            m = bitboard & (bitboard >> shift)
            if m & (m >> (2 * shift)):
                return True
        return False

    def get_winner(self) -> int | None:
        """Return winner ID (0 or 1) or None if no winner"""
        if self.win(opponent=False):
            return self.cur_player
        if self.win(opponent=True):
            return 1 - self.cur_player
        return None

    def get_valid_actions(self) -> list[Action]:
        """Get all legal actions from current state"""
        actions = []

        for col in range(COLS):
            # Drop: column is valid if it has any empty space
            empty_in_col = (~self.get_occupied()) & self._col_mask(col)
            if empty_in_col != 0:
                actions.append(('drop', col))

            # Pop: valid if bottom bit belongs to current player
            bottom_bit = 1 << (col * 7)
            if self.player & bottom_bit:
                actions.append(('pop', col))
        return actions

    def _col_mask(self, col: int) -> int:
        """Create mask for a column"""
        # (1 << 6) - 1 = 111111 (6 bits)
        return ((1 << 6) - 1) << (col * 7)

    def drop_piece(self, col: int) -> tuple[bool, int | None]:
        """
        Drop piece like in Connect 4

        Args:
            col: column to drop piece

        Returns:
            (ValidMove, WinnerAfterMove)
        """
        # Find lowest empty row in column
        col_mask = self._col_mask(col)
        occupied = self.get_occupied()
        empty_in_col = (~occupied) & col_mask

        if empty_in_col == 0:
            return (False, None)  # Column full

        # Get lowest empty bit (closest to bottom)
        # -x = ~x + 1
        # x = [bits] 1 [0's]
        # -x = [~bits] 1 [0's]
        move_bit = empty_in_col & -empty_in_col
        self.player |= move_bit

        if self.win():
            return (True, self.cur_player)

        self.switch_turn()
        self._record_state()
        return (True, None)

    def popout_piece(self, col: int) -> tuple[bool, int | None]:
        """
        Pop out the player's own piece from the bottom of a column.
        The piece is removed and all pieces above it fall down.

        Args:
            col: Column index (0-6) to pop from

        Returns:
            (ValidMove, WinnerAfterMove)
        """
        # Check if column is empty
        col_mask = self._col_mask(col)
        occupied = self.get_occupied()

        # Check if there's any piece in this column
        if (occupied & col_mask) == 0:
            return (False, None)  # Column is empty

        # Get the bottom-most piece in the column
        bottom_bit = 1 << (col * 7)  # Row 0 is bottom

        # Check if the bottom piece belongs to the current player
        if not (self.player & bottom_bit):
            return (False, None)  # Bottom piece is not owned by current player

        # Remove the bottom piece
        self.player &= ~bottom_bit

        # Shift all pieces above down by one position
        # For each row from row 1 to row 5, move the piece down one row
        for row in range(1, 6):  # Start from row 1 (second from bottom)
            current_bit = 1 << (col * 7 + row)
            below_bit = 1 << (col * 7 + (row - 1))

            # Check if there's a piece in current position
            if self.player & current_bit:
                # Move player's piece down
                self.player &= ~current_bit
                self.player |= below_bit
            elif self.opponent & current_bit:
                # Move opponent's piece down
                self.opponent &= ~current_bit
                self.opponent |= below_bit

        if self.win():
            return (True, self.cur_player)
        elif self.win(opponent=True):
            return (True, 1 - self.cur_player)

        self.switch_turn()
        self._record_state()
        return (True, None)

    def get_draw_status(self) -> str | None:
        """
        Return the current draw status

        Returns:
            "BOARD_FULL": Board is full --- player decides
            "THREE_FOLD": Three-fold repetition occured --- either player can claim
            None: No draw available
        """
        if self.get_empty() == 0:
            return "BOARD_FULL"
        elif self.is_threefold_repetition():
            return "THREE_FOLD"
        return None
    
    def get_raw_hash(self) -> int:
        """Returns raw (non-canonical) hash without mirror normalization"""
        return self.zobrist._compute_hash(self.player, self.opponent, self.cur_player)

## 3. Monte Carlo Tree Search

Before implementing the Monte Carlo Tree Search algorithm, we designed two auxiliary tools built for easier convergence and lighter computational effort. 

#### 3.1. Transposition table

A transposition table (TT) is a cache that stores previously seen game positions and their associated statistics so that the search does not re-evaluate the same position multiple times. We used two seperate TTs, one for nodes (board states) and one for edges (state-action pairs).

In [ ]:
from abc import ABC, abstractmethod
from typing import Generic, TypeVar, Callable
import tqdm
import math
import random

K = TypeVar('K')

Action = tuple[str, int]
class BaseEntry(ABC, Generic[K]):
    """TT entry ABC"""
    @property
    @abstractmethod
    def key(self) -> K:
        pass

E = TypeVar('E', bound=BaseEntry)

class EdgeEntry(BaseEntry[tuple[int, Action]]):
    """
    Entry in edge transposition table.
    Notation:
        N(s, a) -> visit count for edge
        W(s, a) -> cumulative total reward
        Q(s, a) -> mean action value: W(s, a) / N(s, a)
    """

    def __init__(self, parent_hash: int, action: Action):
        self._key = (parent_hash, action)
        self.N = 0
        self.W = 0.0

    @property
    def key(self) -> tuple[int, Action]:
        return self._key

    @property
    def Q(self):
        if self.N == 0:
            return 0.0
        return self.W / self.N

    def update(self, result: float) -> None:
        """Update statistics"""
        self.N += 1
        self.W += result

class NodeEntry(BaseEntry[int]):
    """
    Entry in node transposition table
    Notation:
        N(s) -> visit count for node
        V(s) -> estimate evaluation of node (heuristic / NN)
        π(s) -> policy at node
        P(s, a) = π(s)[a] -> prior probability of action
    """

    def __init__(self, state_hash: int,
                 children: dict[Action, int] = None,
                 priors: dict[Action, float] = None):
        self._key = state_hash
        self.N: int = 0
        self.V: float = 0.0
        self.P: dict[Action, float] = priors or {}
        self.children: dict[Action, int] = children or {}
        self.is_terminal: bool = False
        self.winner = None

    @property
    def key(self):
        return self._key

    def update(self) -> None:
        self.N += 1

class TranspositionTable(Generic[K, E]):
    """Hash table for storing unique states"""

    def __init__(self, factory: Callable[[K], E]):
        self.table: dict[K, E] = {}
        self._factory = factory

    def __len__(self):
        return len(self.table)

    def get(self, key) -> E | None:
        return self.table.get(key)

    def put(self, key) -> E:
        if key not in self.table:
            self.table[key] = self._factory(key)
        return self.table[key]

#### 3.2. Fast heuristic

This class precomputes all possible four-in-a-row bitmasks at initialisation time. Before running any simulations, MCTS checks whether the current player can win immediately or whether the opponent has as immediate winning threat that needs to be blocked. This prevents the algorithm to search positions where the correct move is obvious.

In [ ]:
Action = tuple[str, int]
class FastHeuristic:
    def __init__(self):
        # Precompute win patterns for each possible position
        self.win_masks = self._precompute_win_masks()

    def _precompute_win_masks(self):
        """Precompute all possible 4-in-a-row masks for faster win detection"""
        masks = []

        # Horizontal: 4 consecutive bits in same row
        for row in range(6):
            for col in range(4):
                mask = 0
                for i in range(4):
                    bit_pos = row + (col + i) * 7
                    mask |= 1 << bit_pos
                masks.append(mask)

        # Vertical: 4 consecutive bits in same column
        for col in range(7):
            for row in range(3):
                mask = 0
                for i in range(4):
                    bit_pos = (row + i) + col * 7
                    mask |= 1 << bit_pos
                masks.append(mask)

        # Diagonal down-right
        for col in range(4):
            for row in range(3):
                mask = 0
                for i in range(4):
                    bit_pos = (row + i) + (col + i) * 7
                    mask |= 1 << bit_pos
                masks.append(mask)

        # Diagonal up-right
        for col in range(4):
            for row in range(3, 6):
                mask = 0
                for i in range(4):
                    bit_pos = (row - i) + (col + i) * 7
                    mask |= 1 << bit_pos
                masks.append(mask)

        return masks

    def _check_win_fast(self, bitboard: int) -> bool:
        """Check win using precomputed masks"""
        for mask in self.win_masks:
            if (bitboard & mask) == mask:
                return True
        return False

    def is_winning_move(self, state: PopOut, action: Action) -> bool:
        """
        Check if making this move results in a win for the current player.
        Uses bitboard operations without modifying the original state.
        """
        action_type, col = action
        player = state.player
        opponent = state.opponent
        occupied = (player | opponent) | state.filler_mask
        
        if action_type == 'drop':
            col_mask = state._col_mask(col)
            empty_in_col = (~occupied) & col_mask
            
            if empty_in_col == 0:
                return False
            
            # Get lowest empty bit (two's complement trick)
            move_bit = empty_in_col & -empty_in_col
            new_player = player | move_bit
            
            return self._check_win_fast(new_player)
        
        elif action_type == 'pop':
            bottom_bit = 1 << (col * 7)
            
            # Check if bottom piece belongs to current player
            if not (player & bottom_bit):
                return False
            
            # Simulate popout by removing bottom piece and shifting
            new_player, _ = self._simulate_popout(player, opponent, col)
            
            return self._check_win_fast(new_player)
        
        return False

    def _simulate_popout(
        self,
        player: int,
        opponent: int,
        col: int) -> tuple[int, int]:
        """
        Simulate a popout move.

        Returns:
            (new_player, new_opponent)
        """

        bottom_bit = 1 << (col * 7)

        # Remove bottom piece from player
        new_player = player & ~bottom_bit
        new_opponent = opponent

        for row in range(1, 6):
            current_bit = 1 << (col * 7 + row)
            below_bit = 1 << (col * 7 + (row - 1))

            if player & current_bit:
                new_player &= ~current_bit
                new_player |= below_bit

            elif opponent & current_bit:
                new_opponent &= ~current_bit
                new_opponent |= below_bit

        return new_player, new_opponent

    def is_blocking_move(self, state: PopOut, action: Action) -> bool:
        """
        A move is blocking if:
        1. Opponent currently has at least one winning move
        2. After our move, opponent has none
        """

        # Does opponent currently threaten a win?
        opponent_has_threat = False

        # Create perspective where opponent becomes current player
        opp_state = state.copy()
        opp_state.switch_turn()

        for opp_action in opp_state.get_valid_actions():
            if self.is_winning_move(opp_state, opp_action):
                opponent_has_threat = True
                break

        if not opponent_has_threat:
            return False

        # Apply our move
        temp_state = state.copy()

        action_type, col = action

        if action_type == 'drop':
            valid, _ = temp_state.drop_piece(col)
        else:
            valid, _ = temp_state.popout_piece(col)

        if not valid:
            return False

        # After our move, opponent should NOT have winning move
        for opp_action in temp_state.get_valid_actions():
            if self.is_winning_move(temp_state, opp_action):
                return False

        return True

Imporant class methods:

- `_precompute_win_masks()` → list[int]

Generates bitmasks for every possible winning line: 24 horizontal, 21 vertical, 12 diagonal down-right, and 12 diagonal up-right lines, for a total of 69 masks. 

- `is_winning_move(state, action)` → bool

Without modifying the game state, simulates the effect of the given action on the player's bitboard and then checks the resulting bitboard against every precomputed win mask. For "drop" actions it uses the two's complement trick to find the landing cell. For "pop" actions it calls _simulate_popout to compute the resulting bitboard layout. Returns True if the player would win immediately.

- `is_blocking_move(state, action)` → bool

Determines whether a move eliminates all of the opponent's immediate winning threats. It first checks if the opponent has at least one winning move by temporarily swapping bitboards (treating the opponent as the current player) and calling is_winning_move. If no opponent threat exists, the method returns False. Otherwise it applies the move to a copy of the state and verifies that the opponent no longer has any winning response. This is used in the MCTS search to immediately prioritise blocking moves.

- `_simulate_popout(player, opponent, col)` → (int, int)

Returns the resulting (new_player, new opponent) bitboards simulated after a popout move. The bottom piece is removed from the player's bitboard, then rows 1–5 are iterated to shift each piece down by one position in both bitboards.



#### 3.3. Monte Carlo Tree Search Implementation

Monte Carlo Tree Search is a best-first search algorithm that uses random simulations to estimate the value of game states. It builds a search tree incrementally, exploring the most promising parts of the tree firstly. 

Each iteration of MCTS consists of four phases:

- <strong> Selection: </strong> The algorithm traverses the existing tree by repeatedly selecting child nodes with the highest Upper Confidence Bound for Trees (UCT) until it reaches a node with unexpanded children or that represents a terminal state.

$$UCT(node) = \frac{U(node)}{N(node)} + C \cdot \sqrt{\frac{\ln N(PARENT(node))}{N(node)}}$$
where:

- $U(node)$ = number of wins for the node
- $N(node)$ = number of times the choice on the node was made
- $N(PARENT(node))$ = number of times the father was visited
- $C$ = we used $\sqrt{2}$

This formula combines two vertens: unexplored noed and nodes with high average win ratio. This approach makes searches more guided while reducing the effect of low-luck rollouts. The UCT for unvisited nodes is infinite.


- <strong> Expansion: </strong> Once a node with unexplored children is found, one of the untried actions is selected at random and applied to produce a new child state. The child is registered in the transposition tables and its node entry is created. The system checks whether the child's canonical hash already exists in the table before allocating a new entry: if it does, the existing entry is reused

- <strong> Rollout: </strong> From the newly expanded node, moves are chosen uniformly at random from the legal move set until a terminal state is reached (win, draw, or a depth limit of 200 moves). The result is returned as +1.0 (win), −1.0 (loss), or 0.0 (draw or depth limit).

- <strong> Backpropagation: </strong> The result of the rollout is propagated back along the path from the leaf to the root. At each edge, the result is negated before being applied

In our implementation, the action to play is determined by the visit counts of the root's children. Visit counts are a more robust estimator because they are less susceptible to outilers resulting from unlucky rollouts. The counts can be normalised into a probability distribution using a temperature parameter τ:
$$τ(a | s) = \frac{N(s, a)^{1/τ}}{\sum_b N(s,b)^{1/τ}}

In [ ]:
EXPLORATION_CONSTANT = 1.414  # sqrt(2)


class MCTSNode:
    """Nodes are shared and can have multiple parents"""
    __slots__ = ['hash']

    def __init__(self, hash: int):
        self.hash = hash


class MCTS:
    """Monte Carlo Tree Search with Transposition Table"""
    __slots__ = ['C', 'max_simulations', 'node_cache', 'node_tt',
                 'edge_tt', 'root', 'root_state', 'root_player', 'heuristic']

    def __init__(self, exploration_constant: float = EXPLORATION_CONSTANT,
                 max_simulations: int = 1000):
        self.C = exploration_constant
        self.max_simulations = max_simulations
        self.node_cache: dict[int, MCTSNode] = {}
        self.node_tt = TranspositionTable[int, NodeEntry](
                lambda key: NodeEntry(key)
        )
        self.edge_tt = TranspositionTable[tuple[int, Action], EdgeEntry](
                lambda key: EdgeEntry(key[0], key[1])
        )
        self.root: MCTSNode | None = None
        self.root_state: PopOut | None = None
        self.root_player: int = None
        self.heuristic = FastHeuristic()

    def search(self, state: PopOut, simulations: int = None,
               show_progress: bool = False) -> dict[Action, float]:
        """
        Run MCTS from given state and return action probabilities.

        Args:
            state: Initial game state
            simulations: Number of simulations to run (uses max_simulations if None)

        Returns:
            Dictionary mapping Action -> probability
        """
        valid = state.get_valid_actions()
        valid_set = set(valid)

        # Immediate win
        for action in valid:
            if self.heuristic.is_winning_move(state, action):
                return {a: (1.0 if a == action else 0.0) for a in valid}

        # Find opponent's winning moves using swapped bitboards (no copy)
        # is_winning_move reads state.player — swap so opponent is "player"
        class SwappedState:
            def __init__(self, s):
                self.player = s.opponent
                self.opponent = s.player
                self.filler_mask = s.filler_mask
                self._col_mask = s._col_mask

        swapped = SwappedState(state)
        for action in valid:  # opponent can only threaten via valid actions too
            if action in valid_set and self.heuristic.is_winning_move(swapped, action):
                return {a: (1.0 if a == action else 0.0) for a in valid}

        # Proceed with regular MCTS
        sim_count = simulations or self.max_simulations

        # Create root node
        root_hash = state.get_hash()
        self.root = MCTSNode(root_hash)
        self.node_cache[root_hash] = self.root
        self.node_tt.put(root_hash)
        self.root_state = state.copy()
        self.root_player = state.cur_player

        # Progress bar
        pbar = tqdm(total=sim_count, desc="MCTS Simulations",
                    unit="sim", disable=not show_progress)

        # Run simulations
        for sim in range(sim_count):
            # Selection + Expansion
            leaf_node, leaf_state, path = self._select(self.root, self.root_state.copy())

            # Simulation (rollout)
            result: float = self._simulate(leaf_state)

            # Backpropagation
            self._backpropagate(leaf_node, path, result)

            if sim == 10:
                print(self.node_tt.get(root_hash).children)

            pbar.update(1)
            if sim % 100 == 0 and sim > 0:
                pbar.set_postfix({
                    'Nodes': len(self.node_cache),
                    'Edges': len(self.edge_tt)
                })
        pbar.close()
        # Return action probabilities
        return self._get_action_probs(self.root, self.root_state)

    def _transition(self, state: PopOut, action: Action, verbose=False) -> PopOut | None:
        """
        T(s, a) = s'.

        Returns:
            New state or None if invalid action
        """
        new_state = state.copy()
        action_type, col = action
        if action_type == 'drop':
            valid, _ = new_state.drop_piece(col)
        else:  # 'pop'
            valid, _ = new_state.popout_piece(col)

        if not valid and verbose:
            print(f"{action} is not valid for the current state:\n{state}")
        return new_state if valid else None

    def _expand(self, node: MCTSNode, action: Action,
                child_state: PopOut, child_hash: int) -> tuple[MCTSNode, bool]:
        """Expand a node and return its child and whether a new edge was registered."""
        child_node = None
        child_entry = self.node_tt.get(child_hash)
        if child_entry:  # child node has been seen before
            child_node = self.node_cache[child_hash]
        else:
            # Create entry and set terminal status
            child_entry = self.node_tt.put(child_hash)
            if not child_entry.is_terminal:
                winner = child_state.get_winner()
                if winner is not None:
                    child_entry.is_terminal = True
                    child_entry.winner = winner

        if child_node is None:
            child_node = MCTSNode(child_hash)
            self.node_cache[child_hash] = child_node

        node_entry = self.node_tt.get(node.hash)

        """
        # Re-use child node if previously expanded
        if action in node_entry.children:
            return child_node, False

        # If mirror action was already expanded it points to the same child hash
        # due to symmetry. Therefore, reuse the edge as to not create ghost entries
        action_type, col = action
        mirror = (action_type, 6 - col)
        if mirror in node_entry.children:
            assert node_entry.children[mirror] == child_hash, \
                    "Mirror action points to different child, i.e., board is not symmetric"
            return child_node, False
        """
        # Check if representative of this equivalence class is already registered.
        # Reuse it as to not create ghost entries
        if child_hash in node_entry.children.values():
            return child_node, False

        # Register edge
        node_entry.children[action] = child_hash
        self.edge_tt.put((node.hash, action))
        return child_node, True

    def _select(self, node: MCTSNode, state: PopOut
                ) -> tuple[MCTSNode, PopOut, list[tuple[int, Action]]]:
        """
        Selection phase: traverse tree using UCT until leaf node.

        Returns:
            (leaf_node, leaf_state, path)
        """
        path: list[tuple[int, Action]] = []
        visited: set[int] = set()  # prevent cycles within one path
        while True:
            visited.add(node.hash)

            node_entry = self.node_tt.get(node.hash)
            assert node_entry is not None
            if node_entry.is_terminal:  # Therefore it is a leaf node
                return node, state, path

            valid_actions: list[Action] = state.get_valid_actions()
            expanded: set[Action] = set(node_entry.children.keys())

            # Find untried actions using class equivalences
            expanded_hashes: set[int] = {
                    child_hash for child_hash in node_entry.children.values()
                    }
            untried: list[Action] = []
            for action in valid_actions:
                if action in expanded:
                    continue
                child_state = self._transition(state, action)
                if child_state.get_hash() not in expanded_hashes:
                    untried.append(action)

            # Prioritize unseen nodes = leaf nodes
            while untried:
                action: Action = random.choice(untried)
                untried.remove(action)

                child_state = self._transition(state, action)
                if child_state is None:
                    raise Exception(f"{action} is not a valid action on the given state")

                child_hash = child_state.get_hash()

                # Don't expand nodes that produce cycles
                if child_hash in visited:
                    continue

                child_node, edge_registered = self._expand(node, action, child_state, child_hash)
                if edge_registered:
                    path.append((node.hash, action))
                return child_node, child_state, path

            # Fully expanded — select best child using UCT
            best_child, best_state, action = self._select_best_child(node, state, visited)

            # Safeguard: all children are terminal or lead to cycles
            if best_child is None:
                return node, state, path

            path.append((node.hash, action))
            node = best_child
            state = best_state

    def ucb1(self, edge: EdgeEntry, N_parent_total: int) -> float:
        if edge.N == 0:
            return float('inf')
        return edge.Q + self.C * math.sqrt(math.log(N_parent_total) / edge.N)

    def _select_best_child(self, node: MCTSNode, state: PopOut,
                           visited: set[int]) -> tuple[MCTSNode, PopOut, Action]:
        """Select best child using UCB1 formula"""
        node_entry = self.node_tt.get(node.hash)
        assert node_entry is not None

        children: dict[Action, int] = node_entry.children
        N_parent_total = sum(
                self.edge_tt.get((node.hash, a)).N
                for a in children.keys()
                if self.edge_tt.get((node.hash, a)) is not None
        )

        best_value = -float('inf')
        best_child = None
        best_state = None
        best_action = None

        for action, child_hash in children.items():
            if child_hash in visited:  # No cycles
                continue

            child_state = self._transition(state, action)
            if child_state is None:
                continue  # edge valid on a different path

            child_entry = self.node_tt.get(child_hash)
            if child_entry and child_entry.is_terminal:
                if child_entry.winner == self.root_player:  # root player just won — take it immediately
                    return self.node_cache[child_hash], child_state, action
                else:  # opponent just won — skip this losing move
                    continue

            edge = self.edge_tt.get((node.hash, action))
            ucb = self.ucb1(edge, N_parent_total)

            if ucb > best_value:
                best_value = ucb
                best_child = self.node_cache[child_hash]
                best_state = child_state
                best_action = action

        return best_child, best_state, best_action

    def _simulate(self, state: PopOut, max_depth: int = 100) -> float:
        """
        Random rollout until terminal state.
        Returns result from perspective of the player to move at the leaf.
        """
        current = state.copy()
        rollout_player = current.cur_player
        depth = 0

        while depth < max_depth:
            winner = current.get_winner()
            if winner is not None:
                return 1.0 if winner == rollout_player else -1.0

            draw_status = current.get_draw_status()
            if draw_status == "THREE_FOLD":
                return 0.0

            valid_actions: list[Action] = current.get_valid_actions()
            if not valid_actions:  # board full, can't pop
                return 0.0

            action = random.choice(valid_actions)
            current = self._transition(current, action)
            depth += 1

        return 0.0  # depth limit reached

    def _backpropagate(self, node: MCTSNode, path: list[tuple[int, Action]], result: float):
        """Backpropagate result up the tree, alternating perspective each edge"""
        leaf_entry = self.node_tt.get(node.hash)
        assert leaf_entry is not None
        leaf_entry.update()

        for parent_hash, action in reversed(path):
            result = -result
            edge = self.edge_tt.get((parent_hash, action))
            assert edge is not None
            edge.update(result)
            node_entry = self.node_tt.get(parent_hash)
            assert node_entry is not None
            node_entry.update()

    def _get_action_probs(self, node: MCTSNode, state: PopOut,
                          temperature: float = 1.0) -> dict[Action, float]:
        """
        Extract action probabilities from visit counts.

        Args:
            node: Current node
            state: Current state
            temperature: Temperature for exploration (1.0 = proportional, 0.0 = greedy)

        Returns:
            Dictionary mapping Action -> probability
        """
        node_entry = self.node_tt.get(node.hash)
        children: dict[Action, int] = node_entry.children
        valid_set: set[Action] = set(state.get_valid_actions())

        visits = {action: self.edge_tt.get((node.hash, action)).N
                  for action in children}
        total_visits = sum(visits.values())

        if total_visits == 0:  # uniform fallback
            return {action: 1.0 / len(valid_set) for action in valid_set}
        elif temperature == 0.0:  # greedy
            best_action = max(visits.items(), key=lambda x: x[1])[0]
            probs = {action: 1.0 if action == best_action else 0.0 for action in children}
        else:
            total_weighted = sum(v ** (1.0 / temperature) for v in visits.values())
            probs = {
                action: (visits[action] ** (1.0 / temperature)) / total_weighted
                for action in children
            }

        print(f"valid_actions = {valid_set}")
        print(f"canonical probs = {probs}")

        # Build G(s) = A(s) / ~ using canonical hashes
        groups: dict[int, list[Action]] = defaultdict(list)
        for action in valid_set:
            child_state = self._transition(state, action)
            groups[child_state.get_hash()].append(action)

        # For each equivalence class, find the registered edge and distribute
        final_probs = {}
        for canonical, actions in groups.items():
            registered = [a for a in actions if a in probs]
            if not registered:
                continue
            assert len(registered) == 1, "There should only be one representative of the class equivalence"
            prob = probs[registered[0]]
            per_action = prob / len(actions)
            for action in actions:
                final_probs[action] = per_action
        return final_probs

    def get_best_move(self, state: PopOut, simulations: int = 800,
                      show_progress: bool = False) -> Action:
        """
        Run MCTS and return the best move.

        Returns:
            Action: (action_type, col)
                e.g., ('drop', 3), ('pop', 1)
        """
        probs: dict[Action, float] = self.search(state, simulations, show_progress=show_progress)
        if not probs:
            return ('drop', 0)
        return max(probs.items(), key=lambda x: x[1])[0]

    def clear(self):
        """Clear TTs and node cache for a new game"""
        self.node_cache: dict[int, MCTSNode] = {}
        self.node_tt = TranspositionTable[int, NodeEntry](
                lambda key: NodeEntry(key)
        )
        self.edge_tt = TranspositionTable[tuple[int, Action], EdgeEntry](
                lambda key: EdgeEntry(key[0], key[1])
        )

Nodes are identified exclusively by hash, and all statistics live in the transposition tables keyed on that hash. This means the same logical node can be pointed to by multiple parent edges without duplicating storage 

The MCTS class mantains the following attirbutes:

- C: exploration constant
- node_tt: TranspositionTable[int, NodeEntry] - maps state has
- edge_tt: TranspositionTable[(hash, Action), EdgeEntry] — maps (parent hash, action)
- node_cache: dict[int, MCTSNode]
- heuristic: FastHeuristic()

Important class methods:

- `search(state, simulations)` → dict[Action, float]

Before running any simulations it performs two short-circuit checks using the heuristic: 

- if the current player has an immediate winning move it returns that move with probability 1.0.

- if the opponent has an immediate winning move that is also legal for the current player it returns that move with probability 1.0 (forced block).

If neither applies, the method initialises the root node in the transposition tables and runs the standard MCTS. It concludes by calling _get_action_probs to compute the final probability distribution.

- `_select(node, state)` → (leaf_node, leaf_state, path)

At each node it computes the set of untried actions as those whose resulting child hash does not yet appear among the node's registered children. If untried actions exist, one is chosen at random and _expand is called. Otherwise the node is considered fully expanded and _select_best_child is called to descend one level further. The path variable accumulates (parent_hash, action) pairs so that _backpropagate can walk back up the tree.


- `is_blocking_move(state, action)` → bool

Determines whether a move eliminates all of the opponent's immediate winning threats. It first checks if the opponent has at least one winning move by temporarily swapping bitboards (treating the opponent as the current player) and calling is_winning_move. If no opponent threat exists, the method returns False. Otherwise it applies the move to a copy of the state and verifies that the opponent no longer has any winning response. This is used in the MCTS search to immediately prioritise blocking moves.

- `_expand(node, action, child_state, child_hash)` → (child_node, bool)

Creates or retrieves the child node and registers the edge in the transposition tables. If a child with the same canonical hash is already registered under a different action at this node, no new edge is created and the existing one is reused.

- `_select_best_child(node, state, visited)` → (child_node, child_state, action)

Computes UCB1 for each expanded child and returns the one with the highest score. Terminal children that are wins for the root player are returned immediately. Terminal children that are losses for the root player are skipped entirely.

- `ucb1(edge, N_parent_total)` → float

Implements the UCB1 formula. Returns +∞ for unvisited edges to guarantee they are explored first.

- `_simulate(state, max_depth=100)` → float

Performs a random rollout from the given state. Moves are selected uniformly at random, draw conditions are detected at each step, and the result is returned as +1.0, −1.0, or 0.0 from the perspective of the player to move at the leaf node.

- `_backpropagate(node, path, result)`

Walks the path in reverse. At each step the result is negated before being written to the edge entry via and to the node entry.

- `_get_action_probs(node, state, temperature=1.0)` → dict[Action, float]

Reads the visit counts from the edge TT for all children of the root node. Applies temperature scaling and normalises to a probability distribution. Multiple actions that map to the same canonical child state share their combined probability equally, ensuring that symmetric move pairs each receive half the total weight.

- `_get_best_move(state, simulations)` → Action

Calls search and returns the action with the hightes probability

- `clear()`

Resets all the transposition tables and the node cache. This is will soon be called between moves during dataset generation.

## 4. ID3 Decision Tree

In [1]:
import math
from collections import Counter
import pandas as pd
import random

def get_entropy(labels):
    if not labels:
        return 0.0
    else:
        entropy = 0.0
        counts = Counter(labels)
        total  = len(labels)
        for count in counts.values():
            if count > 0:
                entropy -= (count / total) * math.log2(count / total)
        return entropy
    
def information_gain(data, labels, attribute_idx):
    ''' Returns the information gain of a certain attirbute'''
    h_c = get_entropy(labels)
    # Group labels by attribute value
    subsets = {}
    for i, row in enumerate(data):
        key = row[attribute_idx]
        subsets.setdefault(key, []).append(labels[i])
    total = len(labels)
    conditional_entropy = 0.0
    for subset_labels in subsets.values():
        probability = len(subset_labels) / total
        conditional_entropy += probability * get_entropy(subset_labels)
    gain = h_c - conditional_entropy
    return gain

class DecisionTreeNode:
    def __init__(self, is_leaf=False, label=None, feature_idx=None):
        self.is_leaf      = is_leaf
        self.label        = label
        self.feature_idx  = feature_idx
        self.children     = {}

def id3(data, labels, feature_indices, depth=0, max_depth=None):
    # Base cases
    # 1 - All the examples in the node have the same label: it is a leaf
    if (len(set(labels))) == 1: 
        return DecisionTreeNode(is_leaf=True, label=labels[0])
    # 2 - No attributes for splitting 
    if not feature_indices:
        majority = Counter(labels).most_common(1)[0][0]
        return DecisionTreeNode(is_leaf=True, label=majority)
    
    # Choosing the attribute with the highest information gain
    best_idx = None
    best_gain = 0.0
    for idx in feature_indices:
        gain = information_gain(data, labels, idx)
        if gain > best_gain:
            best_gain = gain
            best_idx = idx
    
    # If no best index found, use majority label
    if best_idx is None:
        majority = Counter(labels).most_common(1)[0][0]
        return DecisionTreeNode(is_leaf=True, label=majority)
    
    node = DecisionTreeNode(feature_idx = best_idx)
    values = set(row[best_idx] for row in data)
    remaining = [idx for idx in feature_indices if idx != best_idx]
    
    # Construct subtree
    for val in values:
        sub_data   = [data[i] for i in range(len(data)) if data[i][best_idx] == val]
        sub_labels = [labels[i] for i in range(len(labels)) if data[i][best_idx] == val]
        # Recursive call
        node.children[val] = id3(sub_data, sub_labels, remaining, depth + 1, max_depth)
    return node

def predict(tree, value):
    node = tree
    while not node.is_leaf:
        if node.feature_idx is None:
            return None
        val = value[node.feature_idx]
        if val not in node.children:
            return None  # valor desconhecido
        node = node.children[val]
    return node.label

def accuracy(tree, data, labels):
    total = len(labels)
    correct = 0
    for i, ex in enumerate(data):
        if predict(tree, ex) == labels[i]:
            correct += 1
    if not labels:
        return 0.0
    else:
        return correct / total

#### Testing the ID3 Decision Tree Implementation on the Iris dataset

In [2]:
df = pd.read_csv('iris.csv')
features = df.columns[1:-1]
label = df.columns[-1]

def learn_bins(data, n_bins = 3):
    '''Learn bin boundaries from data'''
    min_val = min(data)
    max_val = max(data)
    bin_size = (max_val - min_val) / n_bins
    return [min_val + bin_size, min_val + 2 * bin_size]

def discretize_value(value, bins):
    '''Discretize a single value using predefined bins'''
    if value <= bins[0]:
        return "low"
    elif value <= bins[1]:
        return "medium"
    else:
        return "high"
    
# Train and test splitting with separate discretization
random.seed(1234)
indices = list(range(len(df)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx = indices[split:]

# Get raw train and test data
train_raw = df.iloc[train_idx]
test_raw = df.iloc[test_idx]

# Learn bins from training data
train_bins = {}
for column in features:
    train_bins[column] = learn_bins(train_raw[column].values)

# Learn bins from test data
test_bins = {}
for column in features:
    test_bins[column] = learn_bins(test_raw[column].values)

# Discretize train_data using train bins
train_data = []
for i in train_idx:
    row = []
    for column in features:
        value = df.loc[i, column]
        discretized_value = discretize_value(value, train_bins[column])
        row.append(discretized_value)
    train_data.append(row)

# Discretize test_data using test bins
test_data = []
for i in test_idx:
    row = []
    for column in features:
        value = df.loc[i, column]
        discretized_value = discretize_value(value, test_bins[column])
        row.append(discretized_value)
    test_data.append(row)

train_labels = [df.loc[i, label] for i in train_idx]
test_labels = [df.loc[i, label] for i in test_idx]

feature_indices = list(range(len(features)))
my_tree = id3(train_data, train_labels, feature_indices)

train_accuracy = accuracy(my_tree, train_data, train_labels)
test_accuracy = accuracy(my_tree, test_data, test_labels)

print(f'\nTrain accuracy: {train_accuracy:.1%}')
print(f'Test accuracy:  {test_accuracy:.1%}')


Train accuracy: 95.8%
Test accuracy:  90.0%


In [ ]:
df = pd.read_csv('popout.csv')

# Removing probability columns
df.drop(columns=["('drop', 0)", "('drop', 1)", "('drop', 2)", "('drop', 3)", "('drop', 4)", "('drop', 5)", "('drop', 6)", "('pop', 0)", 
                 "('pop', 1)", "('pop', 2)", "('pop', 3)", "('pop', 4)", "('pop', 5)", "('pop', 6)"], inplace=True)
df.columns
# New aggreagted dataset
state_cols = [c for c in df.columns if c != "best_action"]
agg = (df.groupby(state_cols, as_index=False)
         .agg({"best_action": lambda x: x.value_counts().idxmax()}))
agg.to_csv("popout_agg.csv", index=False)

In [ ]:
import ast
df_new = pd.read_csv("popout_agg.csv")
label = "best_action"
state_cols = [c for c in df_new.columns if c != label]

# Aggregating: 2 top states for accuracy measure
top2_map = {}
for key, group in df.groupby(state_cols):
    top2 = [action for action, _ in Counter(group["best_action"].tolist()).most_common(2)]
    top2_map[key] = top2

def top2_accuracy(tree, data, state_cols_values, top2_map):
    correct = 0
    for i, row in enumerate(data):
        predicted = predict(tree, row)
        key = tuple(state_cols_values[i])  # must match the groupby key
        top2 = top2_map.get(key, [])
        if predicted in top2:
            correct += 1
    return correct / len(data) if data else 0.0

def str_to_action(s):
    if s is None:
        return None

    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return None

def decode_state(row, col_names):
    """Rebuild a PopOut game from encoded feature row"""
    game = PopOut()
    for idx, val in enumerate(row):
        col = idx // 6
        r = idx % 6
        bit_index = col * 7 + r
        if val == "player":
            game.player |= (1 << bit_index)
        elif val == "opponent":
            game.opponent |= (1 << bit_index)
    return game

def valid_move_accuracy(tree, data):
    correct = 0
    for row in data:
        predicted = predict(tree, row)
        if predicted is None:
            continue
        predicted = str_to_action(predicted)
        game = decode_state(row, state_cols)
        if predicted in game.get_valid_actions():
            correct += 1
    return correct / len(data) if data else 0.0

random.seed(1234)
indices = list(range(len(df_new)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx = indices[split:]

train_labels = [df_new.loc[i, label] for i in train_idx]
test_labels = [df_new.loc[i, label] for i in test_idx]

train_data = df_new.loc[train_idx, state_cols].values.tolist()
test_data  = df_new.loc[test_idx,  state_cols].values.tolist()

feature_indices = list(range(len(state_cols)))
my_tree = id3(train_data, train_labels, feature_indices)
test_states = df_new.loc[test_idx, state_cols].values.tolist()

top2_acc = top2_accuracy(my_tree, test_data, test_states, top2_map)
moves_acc = valid_move_accuracy(my_tree, test_data)
print(f"Valid move accuracy: {moves_acc:.1%}")
print(f"Top-2 accuracy: {top2_acc:.1%}")

## Generating the dataset

In [ ]:
import csv
import random
from src.game import PopOut, ROWS, COLS
from src.mcts import MCTS
 
def encode_state(game: PopOut):
    features = []
    for row in range(ROWS):
        for col in range(COLS):
            bit_index = col * 7 + row
            mask = 1 << bit_index
            if game.player & mask:
                features.append("player")
            elif game.opponent & mask:
                features.append("opponent")
            else:
                features.append("empty")
    return features
 
def action_to_str(action):
    return f"{action[0]}_{action[1]}"
 
def str_to_action(s):
    parts = s.split("_")
    return (parts[0], int(parts[1]))
 
def self_play_game(mcts: MCTS, simulations: int = 800):
    """
    Play one game to completion using MCTS.
    At each step:
      - run search(state, simulations)
      - record the full probability distribution (probs) over all actions
      - sample next move from the returned PMF
      - clear mcts between moves
 
    Returns list of (state_hash, game_copy_at_that_state, probs)
    for every move, where probs is the full MCTS visit distribution.
    """
    game = PopOut()
    ply = []  # (state_hash, state_snapshot, chosen_action)
 
    max_moves = 200
    for _ in range(max_moves):
        if game.get_winner() is not None:
            break
        valid = game.get_valid_actions()
        if not valid:
            break
 
        probs = mcts.search(game, simulations=simulations)
 
        # Sample next move from the full distribution
        actions = list(probs.keys())
        weights = [probs[a] for a in actions]
        chosen = random.choices(actions, weights=weights, k=1)[0]
 
        # Store snapshot + full probs
        ply.append((game.zobrist._compute_hash(game.player, game.opponent, game.cur_player), game.copy(), chosen))
 
        # Apply move
        atype, col = chosen
        if atype == 'drop':
            valid_move, winner = game.drop_piece(col)
        else:
            valid_move, winner = game.popout_piece(col)
        assert valid_move is not None
 
        mcts.clear()
 
        if winner is not None:
            break
 
    return ply

def generate_dataset(n_games: int = 100,
                     simulations: int = 800,
                     output_path: str = 'popout_dataset.csv'):
    mcts = MCTS(exploration_constant=1.414, max_simulations=simulations)
    ply = []
 
    for game_idx in range(n_games):
        print(f"\n=== Game {game_idx + 1}/{n_games} ===")
        game_ply = self_play_game(mcts, simulations=simulations)
        ply.extend(game_ply)
        mcts.clear()

    # Build CSV
    feature_cols = []
    for col in range(COLS):
        for row in range(ROWS):
            feature_cols.append(f"cell_r{row}c{col}")
    header = feature_cols + ["best_action"]
 
    written = 0
    with open(output_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)

        for _, state, action in ply:
            X = encode_state(state)
            Y = action_to_str(action)
            #for features, action in zip(X, Y):
            #    writer.writerow([*features, action])
            writer.writerow([*X, action])
            written += 1
 
    print(f"\nDataset written to '{output_path}' — {written} rows.")
    return output_path
 
if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description="Generate PopOut MCTS dataset")
    parser.add_argument("--games",   type=int, default=50,
                        help="Number of self-play games (default: 50)")
    parser.add_argument("--sims",    type=int, default=800,
                        help="MCTS simulations per move (default: 800)")
    parser.add_argument("--output",  type=str, default="popout_dataset.csv",
                        help="Output CSV path")
    parser.add_argument("--seed",    type=int, default=42)
    args = parser.parse_args()
    random.seed(args.seed)
    out = generate_dataset(n_games=args.games,
                           simulations=args.sims,
                           output_path=args.output)